In [1]:
!pip install opendatasets

In [2]:
import opendatasets as od

In [3]:
dataset_link = "https://www.kaggle.com/datasets/nih-chest-xrays/data/data"
od.download(dataset_link)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: aldogianfranco
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data


100%|██████████| 42.0G/42.0G [32:23<00:00, 23.2MB/s]


In [4]:
import os
os.chdir("data")
os.listdir()

['test_list.txt',
 'ARXIV_V5_CHESTXRAY.pdf',
 'images_009',
 'README_CHESTXRAY.pdf',
 'images_001',
 'images_004',
 'images_006',
 'images_011',
 'images_007',
 'images_010',
 'images_003',
 'images_002',
 'images_008',
 'train_val_list.txt',
 'FAQ_CHESTXRAY.pdf',
 'BBox_List_2017.csv',
 'LOG_CHESTXRAY.pdf',
 'Data_Entry_2017.csv',
 'images_005',
 'images_012']

In [5]:
import pandas as pd
archivo = "Data_Entry_2017.csv"
df = pd.read_csv(archivo)
df.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [33]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.preprocessing import MultiLabelBinarizer
from tensorflow.keras.metrics import AUC
import numpy as np
import pandas as pd

In [7]:
# Cargar etiquetas
dfe = pd.read_csv("Data_Entry_2017.csv")
train_files = set(open("train_val_list.txt").read().splitlines())
test_files = set(open("test_list.txt").read().splitlines())

In [8]:
# Obtener lista de todas las imágenes en subdirectorios
image_paths = {}
for root, dirs, files in os.walk("."):
    if "images" in dirs:  # Buscar dentro del subdirectorio 'images'
        images_path = os.path.join(root, "images")
        for file in os.listdir(images_path):
            if file.endswith(".png") or file.endswith(".jpg") or file.endswith(".jpeg"):
                image_paths[file] = os.path.join(images_path, file)

In [9]:
# Convertir las etiquetas separadas por '|' en listas
dfe["Finding Labels List"] = dfe["Finding Labels"].apply(lambda x: x.split("|") if isinstance(x, str) else [])

In [10]:
# Aplicar binarización
mlb = MultiLabelBinarizer()
labels_matrix = mlb.fit_transform(dfe["Finding Labels List"])
label_columns = mlb.classes_

In [11]:
labels_df = pd.DataFrame(labels_matrix, columns=label_columns)
dfe = pd.concat([dfe, labels_df], axis=1)

In [12]:
# Agregar la ruta completa de las imágenes
dfe["full_path"] = dfe["Image Index"].map(image_paths)

In [13]:
# Filtrar solo las imágenes existentes
dfe = dfe[dfe["full_path"].notna()]

In [14]:
# Generadores de imágenes
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

In [15]:
train_gen = datagen.flow_from_dataframe(
    dfe[dfe["Image Index"].isin(train_files)],
    x_col="full_path", y_col=label_columns,
    target_size=(224, 224), batch_size=32, class_mode="raw",
    subset="training")

Found 69220 validated image filenames.


In [16]:
val_gen = datagen.flow_from_dataframe(
    dfe[dfe["Image Index"].isin(train_files)],
    x_col="full_path", y_col=label_columns,
    target_size=(224, 224), batch_size=32, class_mode="raw",
    subset="validation")

Found 17304 validated image filenames.


In [30]:
test_gen = datagen.flow_from_dataframe(
    dfe[dfe["Image Index"].isin(test_files)],
    x_col="full_path",
    y_col=label_columns.tolist(),
    target_size=(224, 224),
    batch_size=32,
    class_mode="raw",
    shuffle=False)

Found 25596 validated image filenames.


In [17]:
print("Etiquetas en train_gen:", dfe[dfe["Image Index"].isin(train_files)][label_columns.tolist()].sum(axis=0))
print("Etiquetas en val_gen:", dfe[dfe["Image Index"].isin(test_files)][label_columns.tolist()].sum(axis=0))

Etiquetas en train_gen: Atelectasis            8280
Cardiomegaly           1707
Consolidation          2852
Edema                  1378
Effusion               8659
Emphysema              1423
Fibrosis               1251
Hernia                  141
Infiltration          13782
Mass                   4034
No Finding            50500
Nodule                 4708
Pleural_Thickening     2242
Pneumonia               876
Pneumothorax           2637
dtype: int64
Etiquetas en val_gen: Atelectasis           3279
Cardiomegaly          1069
Consolidation         1815
Edema                  925
Effusion              4658
Emphysema             1093
Fibrosis               435
Hernia                  86
Infiltration          6112
Mass                  1748
No Finding            9861
Nodule                1623
Pleural_Thickening    1143
Pneumonia              555
Pneumothorax          2665
dtype: int64


In [18]:
# Modelo preentrenado
base_model = DenseNet121(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True  # Descongelar todo
for layer in base_model.layers[:-50]:  # Mantener congeladas las primeras capas
    layer.trainable = False

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [19]:
# Actualizar num_classes
num_classes = len(label_columns)

In [20]:
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation="relu")(x)
out = Dense(num_classes, activation="sigmoid")(x)  # Sigmoid para multilabel

In [65]:
model = Model(inputs=base_model.input, outputs=out)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=[ROC()])

In [ ]:
# Entrenamiento
model.fit(train_gen, validation_data=val_gen, epochs=1)

In [26]:
loss, auc = model.evaluate(val_gen)
print(f"Pérdida en validación: {loss:.4f}")
print(f"AUC en validación: {auc:.4f}")

541/541 ━━━━━━━━━━━━━━━━━━━━ 266s 491ms/step - auc: 0.9027 - loss: 0.1674
Pérdida en validación: 0.1681
AUC en validación: 0.9027


In [27]:
print("Loss function:", model.loss)
print("Optimizer:", model.optimizer)
print("Metrics:", model.metrics)

Loss function: binary_crossentropy
Optimizer: <keras.src.optimizers.adam.Adam object at 0x7ca490569e90>
Metrics: [<Mean name=loss>, <CompileMetrics name=compile_metrics>]


In [31]:
loss, auc = model.evaluate(test_gen)
print(f"Pérdida en testing: {loss:.4f}")
print(f"AUC en testing: {auc:.4f}")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


800/800 ━━━━━━━━━━━━━━━━━━━━ 422s 528ms/step - auc: 0.8171 - loss: 0.2561
Pérdida en validación: 0.2452
AUC en validación: 0.8370


In [32]:
predictions = model.predict(test_gen)

800/800 ━━━━━━━━━━━━━━━━━━━━ 398s 483ms/step


In [61]:
threshold = 0.4  # Puedes probar con 0.3 o 0.4 si el modelo predice pocas etiquetas
binary_predictions = (predictions > threshold).astype(int)

In [62]:
for i in range(20):
    img_path = test_gen.filepaths[i]  # Ruta de la imagen
    true_labels = test_gen.labels[i]  # Etiquetas reales (binarizadas)
    pred_labels = binary_predictions[i]  # Predicciones del modelo

    print(f"Imagen: {os.path.basename(img_path)}")
    print("Etiquetas reales: ", [label_columns[j] for j in np.where(true_labels == 1)[0]])
    print("Etiquetas predichas:", [label_columns[j] for j in np.where(pred_labels == 1)[0]])
    print("-" * 50)

Imagen: 00000003_000.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: ['No Finding']
--------------------------------------------------
Imagen: 00000003_001.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: ['No Finding']
--------------------------------------------------
Imagen: 00000003_002.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: ['No Finding']
--------------------------------------------------
Imagen: 00000003_003.png
Etiquetas reales:  ['Hernia', 'Infiltration']
Etiquetas predichas: []
--------------------------------------------------
Imagen: 00000003_004.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: ['Atelectasis']
--------------------------------------------------
Imagen: 00000003_005.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: []
--------------------------------------------------
Imagen: 00000003_006.png
Etiquetas reales:  ['Hernia']
Etiquetas predichas: ['No Finding']
--------------------------------------------------
Imagen: 00000

In [63]:
from sklearn.metrics import classification_report

# Obtener etiquetas reales
true_labels = test_gen.labels  # Ya están en formato binario

# Calcular métricas
report = classification_report(true_labels, binary_predictions, target_names=label_columns)
print(report)

                    precision    recall  f1-score   support

       Atelectasis       0.42      0.10      0.16      3279
      Cardiomegaly       0.48      0.01      0.02      1069
     Consolidation       0.00      0.00      0.00      1815
             Edema       0.00      0.00      0.00       925
          Effusion       0.48      0.32      0.39      4658
         Emphysema       0.35      0.09      0.14      1093
          Fibrosis       0.00      0.00      0.00       435
            Hernia       0.00      0.00      0.00        86
      Infiltration       0.44      0.12      0.19      6112
              Mass       0.41      0.02      0.04      1748
        No Finding       0.61      0.53      0.56      9861
            Nodule       0.49      0.02      0.04      1623
Pleural_Thickening       0.00      0.00      0.00      1143
         Pneumonia       0.00      0.00      0.00       555
      Pneumothorax       0.52      0.02      0.04      2665

         micro avg       0.54      0.2

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [64]:
from sklearn.metrics import accuracy_score

# Comparar predicciones exactas con etiquetas reales
accuracy = accuracy_score(true_labels, binary_predictions)
print(f"Exact Match Accuracy: {accuracy:.4f}")

Exact Match Accuracy: 0.2336
